<a href="https://colab.research.google.com/github/AditiAdhikari05/arxivist/blob/notebook/workspace/paper-repos/arxiv_2511_19942/notebooks/reproduce_arxiv_2511_19942.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Differential Smoothing Mitigates Sharpening and Improves LLM Reasoning
**ArXivist-generated reproduction notebook**

Paper: [arXiv:2511.19942](https://arxiv.org/abs/2511.19942)

This notebook walks through the key components of the implementation, runs a small-scale
training loop comparing vanilla GRPO against this paper's DS-GRPO on the Countdown puzzle task,
and verifies the setup matches the paper's described mechanism on a mini scale.

**Scope note:** this reproduction uses Qwen2.5-0.5B-Instruct (paper used 3B) and ~300 training
steps (paper trains "until reward saturates", no fixed count given) to fit a free-tier Colab/
Kaggle GPU. See the repo README for full reasoning on this scope-down.

In [6]:
!git clone https://github.com/AditiAdhikari05/arxivist.git
%cd /content/arxivist/workspace/paper-repos/arxiv_2511_19942


Cloning into 'arxivist'...
remote: Enumerating objects: 2748, done.
remote: Counting objects: 100% (244/244), done.
remote: Compressing objects: 100% (199/199), done.
remote: Total 2748 (delta 78), reused 87 (delta 37), pack-reused 2504 (from 2)
Receiving objects: 100% (2748/2748), 14.69 MiB | 24.39 MiB/s, done.
Resolving deltas: 100% (495/495), done.
/content/arxivist/workspace/paper-repos/arxiv_2511_19942


In [7]:
# Check Python version, GPU availability, and key dependencies
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU — training will be very slow, GPU strongly recommended (Kaggle free T4/P100)")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cpu
CUDA available: False
Running on CPU — training will be very slow, GPU strongly recommended (Kaggle free T4/P100)


In [12]:
# Install the project in editable mode (run once)
import subprocess
result = subprocess.run(["pip", "install", "-e", ".."], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)

    error: subprocess-exited-with-error
    
    × python setup.py develop did not run successfully.
    │ exit code: 1
    ╰─> See above for output.
    
    note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× python setup.py develop did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.



## What this paper is about

RL fine-tuning of LLMs (e.g. GRPO) tends to make models **"sharpen"** — collapse onto one narrow
answer pattern, losing diversity of correct approaches, sometimes even converging on wrong answers
confidently. The paper proves *why* this happens (a selection-and-reinforcement bias) and proposes
**differential smoothing**: shift reward down for trajectories the model is already very confident
about *and* correct, and up for trajectories it's unsure about *and* incorrect — keeping
correctness high while preventing collapse.

We test this on **Countdown**: given 3-4 integers, write an expression using each exactly once
that reaches a target number.

## Component 1 — GRPO group-relative advantage (Eq. 5)

Standard GRPO: sample G completions per prompt, then normalize each reward against the group's own
mean/std:
$$A_i = \frac{r_i - \mu(\{r_j\}_{j=1}^G)}{\sigma(\{r_j\}_{j=1}^G)}$$
This is shared by both the baseline and the paper's method — not the novel part.

In [11]:
import sys; sys.path.insert(0, "../src")
from diffsmooth.training.grpo_advantage import GRPOAdvantage

# Toy example: 4 prompts, group size 8, rewards mostly 0 (wrong) with a couple 1s (correct)
toy_rewards = torch.tensor([[1,0,0,0,1,0,0,0], [0,0,0,0,0,0,0,1], [1,1,0,0,0,0,0,0], [0,0,0,0,0,0,0,0]], dtype=torch.float32)
adv_fn = GRPOAdvantage()
advantages = adv_fn.compute(toy_rewards)
print("Input shape:  ", toy_rewards.shape)
print("Output shape: ", advantages.shape)
print("Advantages:\n", advantages)

ModuleNotFoundError: No module named 'diffsmooth'

## Component 2 — Differential Smoothing (Eq. 6) — the paper's actual contribution

$$A_{i}^{\text{DS}}=A_{i}+\begin{cases}-\gamma_{p}\log\pi_{\theta_{\text{old}}}(y_{i}|x),&\text{if }r_{i}=1\\+\gamma_{n}\log\pi_{\theta_{\text{old}}}(y_{i}|x),&\text{otherwise}\end{cases}$$

Correct trajectories the model was already very confident about (high log-prob) get penalized
slightly; incorrect trajectories it was unsure about get boosted slightly. This is what fights
sharpening.

In [ ]:
from diffsmooth.rewards.differential_smoothing import DifferentialSmoothingShaper

shaper = DifferentialSmoothingShaper(gamma_p=0.03, gamma_n=0.01)  # values given in paper for Countdown
is_correct = (toy_rewards > 0).float()
# toy log-probs: correct trajectories the model is very confident about (less negative),
# incorrect ones it's unsure about (more negative)
toy_ref_logprobs = torch.where(is_correct.bool(), torch.tensor(-2.0), torch.tensor(-8.0))

shaped = shaper.shape_advantage(advantages, is_correct, toy_ref_logprobs)
print("Original advantages:\n", advantages)
print("DS-shaped advantages:\n", shaped)
print("\nNotice: confident-correct entries get pulled down slightly, unsure-incorrect pulled up slightly.")

## Component 3 — Countdown verifier

Checks the model's output is a valid expression using exactly the given numbers once each, and
reaches the target. Uses Python's `ast` module to safely parse arithmetic — never `eval()` on
untrusted model output.

In [ ]:
from diffsmooth.rewards.countdown_reward import CountdownVerifier

verifier = CountdownVerifier()
numbers, target = [4, 7, 2], 15

print("Correct completion:  ", verifier.score(numbers, target, "(4 + 7) + 2 + 2"))  # wrong: uses 2 twice -> 0.0
print("Correct completion:  ", verifier.score(numbers, target, "4 + 7 + 2"))          # 13, not 15 -> 0.0
print("Correct completion:  ", verifier.score(numbers, target, "(4 + 7) + 2 + 2"))
print("Actually correct:    ", verifier.score(numbers, target, "7 + 4 + 2 + 2"))       # uses 2 twice, still wrong
print("Valid solution:      ", verifier.score([4, 7, 2], 13, "4 + 7 + 2"))            # target=13 matches

## Mini-training demo: vanilla GRPO vs. DS-GRPO

Runs a **small** number of steps on **synthetic, procedurally-generated** Countdown puzzles (no
downloads needed) — just enough to show the training loop working and losses moving, not to
reproduce the paper's final numbers.

In [ ]:
from diffsmooth.data.countdown_dataset import CountdownDataset

dataset = CountdownDataset(size=100, num_range=[1, 99], num_operands=[3, 4], seed=42)
print(f"Generated {len(dataset)} synthetic Countdown puzzles")
print("Example:", dataset[0])

In [ ]:
from diffsmooth.models.policy import PolicyModel
from diffsmooth.training.trainer import DSGRPOTrainer

config = {
    "gamma_p": 0.03, "gamma_n": 0.01, "group_size": 4, "learning_rate": 1e-6,
    "clip_epsilon_low": 0.20, "clip_epsilon_high": 0.25, "temperature": 0.7,
    "max_new_tokens": 64, "kl_beta": 0.01, "use_differential_smoothing": True,
}

policy = PolicyModel(checkpoint="Qwen2.5-0.5B-Instruct", use_lora=True, load_in_4bit=torch.cuda.is_available(), device=str(device))
n_params = sum(p.numel() for p in policy.model.parameters())
print(f"Model loaded — {n_params/1e6:.1f}M parameters (LoRA + 4-bit if CUDA available)")

trainer = DSGRPOTrainer(policy, verifier, config)

In [ ]:
# Small demo run: 5 steps, batch size 2 (increase for a real run — see train.py for the full script)
batch_size = 2
for step in range(5):
    batch = [dataset[i] for i in range(step * batch_size, step * batch_size + batch_size)]
    metrics = trainer.train_step(batch)
    print(f"step {step} | loss {metrics['loss']:.4f} | mean_reward {metrics['mean_reward']:.3f} | pass@1 {metrics['pass_at_1']:.3f}")

If loss is moving and reward isn't stuck at exactly 0 after a few steps, the training loop is
wired up correctly. For a real comparison, run `train.py` for the full ~300 steps with
`--use_differential_smoothing true` and `false` separately, then compare Pass@K between the two.

## Paper's claimed results (for comparison once you have real numbers)

In [ ]:
paper_results = {
    "dataset": "Countdown",
    "metric": "Pass@K improvement over vanilla GRPO",
    "reported_value": "~4% across all K",
    "baseline": "vanilla GRPO",
    "note": "paper also reports a 4x inference speedup (Pass@64 of vanilla matched at k=16)",
}
print("Paper's claimed results (Countdown task):")
for k, v in paper_results.items():
    print(f"  {k}: {v}")
print("\nTo reproduce properly: run train.py with both --use_differential_smoothing true/false")
print("for the full ~300 steps, then evaluate.py both checkpoints and compare Pass@K.")
print("Then feed your results back to ArXivist's Results Comparator (Stage 6).")

## What to do next

1. **Full training runs (both variants)**:
   ```bash
   python train.py --config configs/config.yaml --use_differential_smoothing false  # baseline
   python train.py --config configs/config.yaml --use_differential_smoothing true   # DS-GRPO
   ```
2. **Evaluation**: `python evaluate.py --checkpoint checkpoints/ds_grpo --config configs/config.yaml`
   (repeat for `checkpoints/vanilla_grpo`)
3. **Compare results**: bring the two evaluation JSONs back to the ArXivist conversation and
   trigger Stage 6 (Results Comparator) to get a formal reproducibility score and hallucination report.

**Implementation notes from the SIR (top assumptions to keep in mind):**
- Training hyperparameters (optimizer, learning rate, step count) are assumed defaults — not stated in the paper (confidence 0.5)
- Model scaled down to 0.5B from the paper's 3B — a deliberate compute-budget substitution (confidence 0.3)
- Whether the paper uses the frozen base model or the previous policy iterate for the smoothing log-prob term is genuinely ambiguous (confidence 0.5) — we implement the latter (Eq. 6's literal form)